# 索引优化

使用小块文本进行检索可以获得更高的精确度，但小块文本缺乏足够的上下文，可能导致大语言模型（LLM）无法生成高质量的答案；而使用大块文本虽然上下文丰富，却容易引入噪音，降低检索的相关性。

LlamaIndex 提出了一种实用的索引策略——句子窗口检索 ： 它在检索时聚焦于高度精确的单个句子，在送入LLM生成答案前，又智能地将上下文扩展回一个更宽的“窗口”，从而同时保证检索的准确性和生成的质量。

# 建立索引

建立索引优很多方式，这里写句子窗口索引和常规索引

## 句子窗口索引

检索时候用小块，送到LLM用大块的上下文

下面使用LLamaIndex官网的示例来实现句子窗口检索

In [ ]:
import os 

from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader

from llama_index.core.node_parser import SentenceWindowNodeParser


In [ ]:
# 把Settings.llm 和 Settings.embed_model 已经预先配置好
# 在后续如果没有指定模型，他就会自动从Settings中读取对应的配置


from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding



Settings.llm = OpenAILike(
    model = os.getenv("AIHUBMIX_MODEL"),
    api_key = os.getenv("AIHUBMIX_API_KEY"),
    api_base =  os.getenv("AIHUBMIX_BASE_URL"),
    is_chat_model=True
)

# 加载Huggingface上的模型ID
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

e:\anaconda3\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bbfss\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# 加载文档
documents = SimpleDirectoryReader(
    input_files=["./data/C3/pdf/IPCC_AR6_WGII_Chapter03.pdf"]
).load_data()

# 句子窗口索引解析参数
# 1. window size = 3 代表前后三个句子共同组成上下文
# 2. window metadata key = window 代表上下文中存储的内容是窗口组成的内容
# 3. original_text_metadata_key="original_text" 代表这个节点存储的文本内容是该节点最开始的文本
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

# 1. 使用sentence splitter 把文档切割成句子列表
# 2、使用build nodes from splitts 把句子列表转化成Text Node，每个 TextNode 的 text 属性就是这个句子的内容
# 3. 然后填充meta信息，其中Node的text内容是节点的内容， 然后窗口的内容是前后三个node共同组成的节点内容
# 4. SentenceWindowNodeParser 最终返回一个 TextNode 列表。列表中的每个节点都代表一个独立的句子，其 text 属性用于精确检索，而其 metadata 中则“隐藏”了用于生成答案的丰富上下文窗口。
sentence_nodes = node_parser.get_nodes_from_documents(documents)


# 根据节点建立索引
sentence_index = VectorStoreIndex(sentence_nodes)


## 常规分块索引

In [ ]:
# 分块解析器
from llama_index.core.node_parser import SentenceSplitter
# 向量索引
from llama_index.core import VectorStoreIndex

# 切割成句子列表
base_parser = SentenceSplitter(chunk_size=512)
# 每个句子都转化成node
base_nodes = base_parser.get_nodes_from_documents(documents)
# 根据每个node建立向量索引
base_index = VectorStoreIndex(base_nodes)



# 建立查询引擎

## 句子窗口查询引擎

In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

# 建构查询引擎
sentence_query_engines = sentence_index.as_query_engine(
    similarity_top_k = 2,
    
    # 后处理引擎，在查询到结果的时候， 后处理引擎会把window text 替换掉text
    node_postprocessors = [
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ]
)



## 常规查询引擎

In [ ]:
ase_query_engine = base_index.as_query_engine(similarity_top_k=2)

NameError: name 'base_index' is not defined

# 执行查找操作

In [ ]:
# 4. 执行查询并对比结果
query = "What are the concerns surrounding the AMOC?"
print(f"查询: {query}\n")

print("--- 句子窗口检索结果 ---")
window_response = sentence_query_engine.query(query)
print(f"回答: {window_response}\n")

print("--- 常规检索结果 ---")
base_response = base_query_engine.query(query)
print(f"回答: {base_response}\n")